# Tutorial 5.2: Application of PRISM to Real Resolution-Induced Incomplete Registration in Human PD Brain

This tutorial applies PRISM to the C1 human Parkinson disease striatum section prepared in Tutorial 5.1. The registered MSI object is indexed by Visium RNA locations, but one-to-one RNA-MSI matching leaves genuine metabolite-unregistered RNA locations because MSI is sampled at lower spatial density.

PRISM treats RNA as the complete source modality and registered RNA-MSI pairs as observed target evidence to identify striatum spatial domains and complete the metabolite field. The native, unaligned MSI object provides the reference Dopamine-DD pattern; because its pixels are not location-matched to metabolite-unregistered RNA locations, completion is interpreted qualitatively rather than as a paired ground-truth benchmark.

In [ ]:
# 0. Environment and imports
from pathlib import Path

import scanpy as sc
import PRISM
from PRISM import (compute_similarity_prior, plot_task2_real_three_panel, preprocess_omics,
                   run_clustering_eval_plot, select_best_device, set_prism_plot_style, set_seed,
                   show_real_missing)
set_prism_plot_style()

### Load data

`Slice_ID` selects a section prepared in Tutorial 5.1. This cell loads complete RNA, the RNA-indexed registered MSI object used for PRISM, and the native-coordinate raw MSI object used to compare the original and completed Dopamine-DD spatial patterns.


In [ ]:
RANDOM_SEED = 2024
set_seed(RANDOM_SEED)
DEVICE = select_best_device()

Slice_ID = "C1" 
DATA_ROOT = Path("Datasets") / "PD human brain"
SLICE_DIR = DATA_ROOT / Slice_ID
SOURCE_H5AD = SLICE_DIR / f"{Slice_ID}_RNA.h5ad"
TARGET_H5AD = SLICE_DIR / f"{Slice_ID}_MSI_reg_annot.h5ad"
RAW_MSI_H5AD = SLICE_DIR / f"{Slice_ID}_MSI_raw_annot.h5ad"
RESULTS_DIR = Path("Results") / "Tutorial5_2_PD_human_brain" / Slice_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_PATH = RESULTS_DIR / f"{Slice_ID}_PD_human_AOT.npz"
RUN_PREFIX = f"{Slice_ID}_PD_human_PRISM"

adata_source_raw = sc.read_h5ad(SOURCE_H5AD)
adata_target_raw = sc.read_h5ad(TARGET_H5AD)
adata_raw_msi = sc.read_h5ad(RAW_MSI_H5AD)
adata_source_raw.var_names_make_unique()
adata_target_raw.var_names_make_unique()
adata_raw_msi.var_names_make_unique()


### Inspecting real resolution-induced incompleteness

`missing` records the RNA-MSI matching outcome: `missing='1'` marks an RNA location with a retained MSI partner, whereas `missing='0'` marks an RNA location without one. These gaps reflect unequal spatial sampling after registration rather than metabolic zeros; unmatched locations have no paired MSI values for feature-wise imputation metrics.


In [ ]:
# Visualize real RNA-MSI coverage
missing_indices, observed_indices = show_real_missing(adata_target_raw, spatial_key="spatial", label_key="missing",
                                                      plot=True, figsize=(4, 4), s=5,
                                                      title=f"{Slice_ID}: real MSI missingness")
print(f"MSI-missing RNA locations: {len(missing_indices)}/{adata_target_raw.n_obs}")

### Preprocessing RNA and MSI

RNA and MALDI-MSI are processed in their respective molecular spaces. The availability mask is retained so zero-filled MSI placeholders at metabolite-unregistered RNA locations are not interpreted as observed metabolic signal. `keep_metabolites` together with `use_keep_metabolites=True` adds the curated SMA paper metabolites to the selected high-variable MSI feature set, ensuring Dopamine-DD and the other annotated metabolites are retained for training and downstream PD visualization.


In [ ]:
# Preprocess RNA and MSI model inputs
adata_source, _ = preprocess_omics(adata_source_raw, modality="RNA", min_cells=10, hvgs=3000, 
                                   missing_key="missing", data_role="source")

adata_target, _ = preprocess_omics(adata_target_raw, modality="MET", hvms=100, missing_key="missing", 
                                   data_role="target", keep_metabolites=list(PRISM.SMA_PAPER_METABOLITES),
                                   use_keep_metabolites=True)

print("RNA shape after preprocessing:", adata_source.shape)
print("MSI shape after preprocessing:", adata_target.shape)

In [ ]:
# Constructing the RNA similarity prior
distance_matrix, _ = compute_similarity_prior(adata_source, adata_target, PRIOR_PATH, device=DEVICE,
                                              covet_k_spatial=6, covet_gene_num=64, spatial_key="spatial",
                                              missing_key="missing")

In [ ]:
# Constructing spatial graphs
PRISM.Cal_Spatial_Net(adata_source, rad_cutoff=1006.6)
PRISM.Stats_Spatial_Net(adata_source)
PRISM.Cal_Spatial_Net(adata_target, rad_cutoff=1006.6)
PRISM.Stats_Spatial_Net(adata_target)

In [ ]:
# Train PRISM on real MSI incompleteness
adata_source_out, adata_target_out = PRISM.train_PRISM(adata_source, adata_target, distance_matrix,
                                                       k_top=5, n_epochs=1000, lr=1e-3,
                                                       output_dir=str(RESULTS_DIR), file_prefix=RUN_PREFIX,
                                                       device=DEVICE, patience=30, min_epochs=200,
                                                       center_drop_rate=0.1, noise=0.0,
                                                       interaction_pca=True)

### Task 1: Spatial-domain identification

Task 1 clusters `PRISM_emb` to delineate C1 spatial domains. The section-specific `new_label` combines brain-region and dopamine annotations from Tutorial 5.1.


In [ ]:
# 8. Identify C1 spatial domains from the PRISM embedding
adata_clustered, domain_metrics = run_clustering_eval_plot(adata_source_out, emb_key="PRISM_emb", label_key="new_label",
                                                           cluster_key="PRISM_mclust", n_clusters=5,
                                                           s=25, use_pca=True, align_labels=True, 
                                                           aligned_key="PRISM_mclust_domain",
                                                           dataset_name="pd_human_brain_c1")

### Task 2: Missing-metabolite imputation

Task 2 compares the native raw Dopamine-DD pattern with the RNA-indexed observed MSI field and the PRISM-completed field. The latter two panels show how predictions extend the observed metabolite distribution across metabolite-unregistered RNA locations.

In [ ]:
# Visualize native, aligned and PRISM-predicted Dopamine-DD
fig, axes = plot_task2_real_three_panel(adata_aligned=adata_target_out, prior_matrix=distance_matrix,
                                               save_files=False, output_dir=RESULTS_DIR, file_prefix=RUN_PREFIX,
                                               adata_unaligned_raw=adata_raw_msi, feature="Dopamine_DD",
                                               show_missing_only=False)